# IEEE-33 DNR Candidate-Ranking Dataset Generator

This notebook replaces topology-class generation with a **switching-optimization dataset**.

Instead of learning:

```text
load scenario -> topology class
```

it generates ranking examples:

```text
load scenario + candidate radial topology -> loss / voltage / loading
```

For each load scenario, the notebook evaluates several physically valid candidate topologies, ranks them by total loss, and marks the minimum-loss candidate. This avoids severe topology-class imbalance and is more scalable to IEEE-69/123 because the model learns a continuous topology-scoring function rather than a fixed class list.

## What was removed / simplified

- No PV feature engineering.
- No EV feature engineering.
- No time/season/weekend feature engineering.
- No PyTorch, SciPy, Numba, Excel dependency, or topology-class target.

## Hard physical constraints

Accepted candidate evaluations must satisfy:

| Quantity | Constraint |
|---|---:|
| Minimum voltage | `min_voltage_pu >= 0.80` |
| Maximum voltage | `max_voltage_pu <= 1.05` |
| Maximum line loading | `<= 100%` |
| Total loss | `<= 600 kW` |
| Topology | connected radial tree |
| Switch deviation | limited branch-exchange candidate pool |

## Output

HDF5 file:

```text
data/ieee33_dnr_ranking_dataset.h5
```

Each group is one load scenario with multiple candidate topologies and ranking labels.


In [ ]:
import json

import pandapower as pp
import pandapower.networks as pn
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import h5py
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

GLOBAL_SEED = 20260622
N_SCENARIOS = 1000
CANDIDATES_PER_SCENARIO = 8
BATCH_SIZE = 32
N_JOBS = -1
H5_PATH = "data/ieee33_dnr_ranking_dataset.h5"
COMPRESSION = "gzip"
COMPRESSION_OPTS = 4

TOPOLOGY_POOL_TARGET = 120
TOPOLOGY_POOL_MAX_ATTEMPTS = 12000
MAX_BRANCH_EXCHANGES = 2

VOLTAGE_HARD_MIN = 0.80
VOLTAGE_NORMAL_MIN = 0.90
VOLTAGE_MAX = 1.05
LOADING_HARD_MAX = 100.0
LOSS_HARD_MAX_KW = 600.0
MIN_REALISTIC_LOSS_KW = 20.0

LOAD_REGIME_POLICY = {"normal": 0.65, "mild_stress": 0.27, "high_stress": 0.08}
LOAD_SPECS = {
    "normal": {"mean": 0.98, "std": 0.030, "clip": (0.90, 1.05), "total_bounds": (0.93, 1.03)},
    "mild_stress": {"mean": 1.11, "std": 0.035, "clip": (1.05, 1.20), "total_bounds": (1.06, 1.16)},
    "high_stress": {"mean": 1.23, "std": 0.030, "clip": (1.20, 1.30), "total_bounds": (1.18, 1.26)},
}

NODE_FEATURE_NAMES = ["p_load_mw", "q_load_mvar", "feeder_depth", "distance_to_substation", "bus_type", "initial_voltage_pu"]
EDGE_FEATURE_NAMES = ["r_ohm", "x_ohm", "switch_status", "impedance_magnitude"]
GLOBAL_FEATURE_NAMES = ["total_load_mw", "num_buses", "num_lines", "active_lines", "topology_deviation"]
LABEL_NAMES = ["total_loss_kw", "min_voltage_pu", "max_line_loading_percent", "voltage_violation_count", "overload_count", "constraint_violation_count"]
SCENARIO_FEATURE_NAMES = ["total_load_mw", "load_ratio", "regime_code"]

REGIME_CODE = {"normal": 0, "mild_stress": 1, "high_stress": 2}
np.random.seed(GLOBAL_SEED)
plt.rcParams["figure.figsize"] = (9, 5)
print(f"Configuration ready | scenarios={N_SCENARIOS:,} | candidates/scenario={CANDIDATES_PER_SCENARIO}")


## 1. IEEE-33 Network and Graph Utilities

The notebook uses `pandapower.networks.case33bw()` directly. Standard IEEE-33 tie branches are ensured and line current limits are normalized for meaningful loading percentages.


In [ ]:
STANDARD_TIE_BRANCH_DATA = [
    {"from_bus": 20, "to_bus": 7, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 8, "to_bus": 14, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 11, "to_bus": 21, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 17, "to_bus": 32, "r_ohm": 0.5, "x_ohm": 0.5},
    {"from_bus": 24, "to_bus": 28, "r_ohm": 0.5, "x_ohm": 0.5},
]

BASE_NET_JSON = None
BASE_LINE_STATUS = None
BASE_TOTAL_LOAD_MW = None
BASE_POWERFLOW_RESULT = None
TOPOLOGY_POOL = None


def _ensure_line_columns(net):
    if "in_service" not in net.line.columns:
        net.line["in_service"] = True
    net.line["in_service"] = net.line["in_service"].fillna(True).astype(bool)
    if "is_tie" not in net.line.columns:
        net.line["is_tie"] = False
    net.line["is_tie"] = net.line["is_tie"].fillna(False).astype(bool)


def _line_mask_for_endpoints(net, from_bus, to_bus):
    from_values = net.line["from_bus"].astype(int).to_numpy()
    to_values = net.line["to_bus"].astype(int).to_numpy()
    return ((from_values == int(from_bus)) & (to_values == int(to_bus))) | ((from_values == int(to_bus)) & (to_values == int(from_bus)))


def _median_or_default(series, default_value):
    if series is None or len(series) == 0:
        return float(default_value)
    value = float(pd.Series(series).dropna().median())
    return value if np.isfinite(value) else float(default_value)


def _normalize_line_current_limits(net, nominal_max_i_ka=0.40):
    if "max_i_ka" not in net.line.columns:
        net.line["max_i_ka"] = nominal_max_i_ka
    max_i = net.line["max_i_ka"].astype(float)
    invalid = (~np.isfinite(max_i)) | (max_i <= 0.0) | (max_i > 5.0)
    net.line.loc[invalid, "max_i_ka"] = float(nominal_max_i_ka)
    return net


def _ensure_ieee33_tie_lines(net):
    _ensure_line_columns(net)
    max_i_ka = _median_or_default(net.line["max_i_ka"] if "max_i_ka" in net.line.columns else None, 0.40)
    c_nf_per_km = _median_or_default(net.line["c_nf_per_km"] if "c_nf_per_km" in net.line.columns else None, 0.0)
    for tie in STANDARD_TIE_BRANCH_DATA:
        mask = _line_mask_for_endpoints(net, tie["from_bus"], tie["to_bus"])
        matching_indices = list(net.line.index[mask])
        if matching_indices:
            net.line.loc[matching_indices, "is_tie"] = True
            net.line.loc[matching_indices, "in_service"] = False
            continue
        line_idx = pp.create_line_from_parameters(
            net,
            from_bus=tie["from_bus"],
            to_bus=tie["to_bus"],
            length_km=1.0,
            r_ohm_per_km=tie["r_ohm"],
            x_ohm_per_km=tie["x_ohm"],
            c_nf_per_km=c_nf_per_km,
            max_i_ka=max_i_ka,
            name=f"tie_{tie['from_bus'] + 1}_{tie['to_bus'] + 1}",
            in_service=False,
        )
        net.line.at[line_idx, "is_tie"] = True
    net.line["in_service"] = net.line["in_service"].astype(bool)
    net.line["is_tie"] = net.line["is_tie"].astype(bool)
    return net


def get_root_bus(net):
    if len(net.ext_grid) > 0 and "bus" in net.ext_grid.columns:
        return int(net.ext_grid.iloc[0]["bus"])
    return int(net.bus.index.min())


def line_is_closed(net, line_idx):
    closed = bool(net.line.at[line_idx, "in_service"])
    if "switch" in net and len(net.switch) > 0:
        switches = net.switch[(net.switch["et"] == "l") & (net.switch["element"].astype(int) == int(line_idx))]
        if len(switches) > 0:
            closed = closed and bool(switches["closed"].fillna(True).astype(bool).all())
    return bool(closed)


def _line_impedance(net, line_idx):
    row = net.line.loc[line_idx]
    length = float(row["length_km"]) if "length_km" in net.line.columns else 1.0
    r = float(row["r_ohm_per_km"]) * length if "r_ohm_per_km" in net.line.columns else 0.0
    x = float(row["x_ohm_per_km"]) * length if "x_ohm_per_km" in net.line.columns else 0.0
    return r, x


def _bus_load_table(net):
    loads = pd.DataFrame(index=net.bus.index, data={"p_load": 0.0, "q_load": 0.0})
    if len(net.load) == 0:
        return loads
    load_df = net.load.copy()
    if "in_service" in load_df.columns:
        load_df = load_df[load_df["in_service"].fillna(True).astype(bool)]
    if len(load_df) == 0:
        return loads
    grouped = load_df.groupby("bus")[["p_mw", "q_mvar"]].sum()
    loads.loc[grouped.index, "p_load"] = grouped["p_mw"].astype(float)
    loads.loc[grouped.index, "q_load"] = grouped["q_mvar"].astype(float)
    return loads


def _initial_voltage_pu_table(net):
    voltage = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.ext_grid) > 0:
        for _, row in net.ext_grid.iterrows():
            vm_pu = float(row["vm_pu"]) if "vm_pu" in net.ext_grid.columns else 1.0
            voltage.loc[int(row["bus"])] = vm_pu
    return voltage


def build_graph(net, active_only=False):
    _ensure_line_columns(net)
    G = nx.Graph()
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    root_bus = get_root_bus(net)
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        G.add_node(
            bus_idx_int,
            p_load=float(load_table.loc[bus_idx, "p_load"]),
            q_load=float(load_table.loc[bus_idx, "q_load"]),
            bus_type=1.0 if bus_idx_int == root_bus else 0.0,
            voltage=float(voltage_table.loc[bus_idx]),
        )
    for line_idx, row in net.line.iterrows():
        switch_status = 1.0 if line_is_closed(net, line_idx) else 0.0
        if active_only and switch_status < 0.5:
            continue
        r, x = _line_impedance(net, line_idx)
        G.add_edge(
            int(row["from_bus"]),
            int(row["to_bus"]),
            line_idx=int(line_idx),
            r=float(r),
            x=float(x),
            impedance=float(np.sqrt(r * r + x * x)),
            switch_status=float(switch_status),
            is_tie=bool(row["is_tie"]),
        )
    return G


def compute_feeder_depth(G, root_bus):
    depth = {int(node): -1 for node in G.nodes}
    if root_bus not in G.nodes:
        return depth
    for node, value in nx.single_source_shortest_path_length(G, int(root_bus)).items():
        depth[int(node)] = int(value)
    return depth


def compute_distance_to_substation(G, root_bus):
    distance = {int(node): np.inf for node in G.nodes}
    if root_bus not in G.nodes:
        return distance
    try:
        lengths = nx.single_source_dijkstra_path_length(G, int(root_bus), weight="impedance")
    except Exception:
        lengths = nx.single_source_shortest_path_length(G, int(root_bus))
    for node, value in lengths.items():
        distance[int(node)] = float(value)
    return distance


def validate_configuration(net):
    G = build_graph(net, active_only=True)
    return len(G.nodes) > 0 and nx.is_connected(G) and nx.is_tree(G)


def load_ieee33():
    net = pn.case33bw()
    _ensure_ieee33_tie_lines(net)
    _normalize_line_current_limits(net, nominal_max_i_ka=0.40)
    if not validate_configuration(net):
        raise ValueError("Base IEEE-33 topology is not radial and connected.")
    return net


## 2. Candidate Topology Pool

The pool is a feasible search space, not a topology class label set. It includes the base topology, all valid one-step branch exchanges, and random valid two-step branch exchanges up to the configured target.


In [ ]:
def _line_status_vector(net):
    return np.array([1 if line_is_closed(net, idx) else 0 for idx in net.line.index], dtype=np.int8)


def _apply_line_status(net, line_status):
    for line_idx, status in zip(net.line.index, np.asarray(line_status, dtype=np.int8)):
        net.line.at[line_idx, "in_service"] = bool(status)
    return net


def topology_signature_from_status(net, line_status):
    edges = []
    for status, (_, row) in zip(np.asarray(line_status, dtype=np.int8), net.line.iterrows()):
        if int(status) == 1:
            edges.append(tuple(sorted((int(row["from_bus"]), int(row["to_bus"])))) )
    return str(abs(hash(tuple(sorted(edges)))))


def switch_config_key_from_status(line_status):
    return ",".join(str(i) for i, status in enumerate(np.asarray(line_status, dtype=np.int8)) if int(status) == 1)


def _cycle_edges_from_nodes(net, cycle_nodes):
    cycle_set = set(int(node) for node in cycle_nodes)
    cycle_edges = []
    for line_idx, row in net.line.iterrows():
        if not line_is_closed(net, line_idx):
            continue
        if int(row["from_bus"]) in cycle_set and int(row["to_bus"]) in cycle_set:
            cycle_edges.append(int(line_idx))
    return cycle_edges


def _register_topology(pool, seen, net, base_status, actions):
    status = _line_status_vector(net)
    signature = topology_signature_from_status(net, status)
    if signature in seen:
        return False
    if not validate_configuration(net):
        return False
    pool.append({
        "topology_id": signature,
        "switch_config_key": switch_config_key_from_status(status),
        "line_status": status.astype(np.int8).tolist(),
        "topology_deviation": int(np.abs(status - base_status).sum()),
        "switch_operations": actions,
    })
    seen.add(signature)
    return True


def _apply_branch_exchange_step(net, rng, base_depth):
    open_lines = [int(idx) for idx in net.line.index if not line_is_closed(net, idx)]
    if len(open_lines) == 0:
        raise ValueError("No open branches available.")
    close_line = int(rng.choice(open_lines))
    net.line.at[close_line, "in_service"] = True
    trial_graph = build_graph(net, active_only=True)
    cycles = nx.cycle_basis(trial_graph)
    if len(cycles) == 0:
        net.line.at[close_line, "in_service"] = False
        raise ValueError("No cycle created by branch closure.")
    close_row = net.line.loc[close_line]
    endpoints = {int(close_row["from_bus"]), int(close_row["to_bus"])}
    selected_cycle = cycles[0]
    for cycle in cycles:
        if endpoints.issubset(set(int(node) for node in cycle)):
            selected_cycle = cycle
            break
    candidates = [idx for idx in _cycle_edges_from_nodes(net, selected_cycle) if idx != close_line]
    if len(candidates) == 0:
        net.line.at[close_line, "in_service"] = False
        raise ValueError("No opening candidates.")
    scores = []
    for line_idx in candidates:
        row = net.line.loc[line_idx]
        depths = [base_depth.get(int(row["from_bus"]), 0), base_depth.get(int(row["to_bus"]), 0)]
        scores.append(max(depths) + 1.0)
    inv = 1.0 / np.asarray(scores, dtype=float)
    probs = inv / inv.sum()
    open_line = int(rng.choice(candidates, p=probs))
    net.line.at[open_line, "in_service"] = False
    if not validate_configuration(net):
        net.line.at[open_line, "in_service"] = True
        net.line.at[close_line, "in_service"] = False
        raise ValueError("Invalid branch exchange.")
    open_row = net.line.loc[open_line]
    return {
        "closed_line": int(close_line),
        "closed_edge": [int(close_row["from_bus"]), int(close_row["to_bus"])],
        "opened_line": int(open_line),
        "opened_edge": [int(open_row["from_bus"]), int(open_row["to_bus"])],
        "cycle_length": int(len(selected_cycle)),
    }


def enumerate_one_step_topologies(base_net, base_status):
    pool = []
    seen = set()
    _register_topology(pool, seen, base_net, base_status, [])
    open_lines = [int(idx) for idx in base_net.line.index if not line_is_closed(base_net, idx)]
    for close_line in open_lines:
        trial = pp.from_json_string(pp.to_json(base_net))
        trial.line.at[close_line, "in_service"] = True
        cycles = nx.cycle_basis(build_graph(trial, active_only=True))
        if len(cycles) == 0:
            continue
        close_row = trial.line.loc[close_line]
        endpoints = {int(close_row["from_bus"]), int(close_row["to_bus"])}
        selected_cycle = cycles[0]
        for cycle in cycles:
            if endpoints.issubset(set(int(node) for node in cycle)):
                selected_cycle = cycle
                break
        for open_line in _cycle_edges_from_nodes(trial, selected_cycle):
            if open_line == close_line:
                continue
            net = pp.from_json_string(pp.to_json(base_net))
            net.line.at[close_line, "in_service"] = True
            net.line.at[open_line, "in_service"] = False
            if validate_configuration(net):
                open_row = net.line.loc[open_line]
                _register_topology(pool, seen, net, base_status, [{
                    "closed_line": int(close_line),
                    "closed_edge": [int(close_row["from_bus"]), int(close_row["to_bus"])],
                    "opened_line": int(open_line),
                    "opened_edge": [int(open_row["from_bus"]), int(open_row["to_bus"])],
                    "cycle_length": int(len(selected_cycle)),
                }])
    return pool, seen


def precompute_topology_pool(base_net, target_count=TOPOLOGY_POOL_TARGET, seed=GLOBAL_SEED):
    rng = np.random.default_rng(int(seed))
    base_status = _line_status_vector(base_net)
    pool, seen = enumerate_one_step_topologies(base_net, base_status)
    base_depth = compute_feeder_depth(build_graph(base_net, active_only=True), get_root_bus(base_net))
    attempts = 0
    while len(pool) < target_count and attempts < TOPOLOGY_POOL_MAX_ATTEMPTS:
        attempts += 1
        net = pp.from_json_string(pp.to_json(base_net))
        actions = []
        try:
            for _ in range(int(rng.choice([1, 2], p=[0.55, 0.45]))):
                actions.append(_apply_branch_exchange_step(net, rng, base_depth))
            _register_topology(pool, seen, net, base_status, actions)
        except Exception:
            continue
    print(f"Topology pool ready | count={len(pool)} | one-step+random attempts={attempts}")
    return pool


## 3. Realistic Load Scenarios

Load scenarios are bounded and calibrated. PV/EV/time features are intentionally removed for this ranking dataset; the learning signal is load distribution plus candidate topology.


In [ ]:
def _bounded_normal(rng, mean, std, low, high, size):
    return np.clip(rng.normal(loc=mean, scale=std, size=size), low, high)


def _enforce_total_load_bounds(scales, load_buses, base_bus_load, total_bounds, clip_bounds):
    low_total, high_total = total_bounds
    low_clip, high_clip = clip_bounds
    weights = np.array([base_bus_load.get(int(bus), 0.0) for bus in load_buses], dtype=float)
    if weights.sum() <= 0:
        return np.clip(scales, low_clip, high_clip)
    weighted_average = float(np.sum(scales * weights) / np.sum(weights))
    adjusted = scales.copy()
    if weighted_average > high_total:
        adjusted *= high_total / weighted_average
    elif weighted_average < low_total:
        adjusted *= low_total / max(weighted_average, 1e-9)
    return np.clip(adjusted, low_clip, high_clip)


def sample_load_regime(rng):
    names = list(LOAD_REGIME_POLICY.keys())
    weights = np.array([LOAD_REGIME_POLICY[name] for name in names], dtype=float)
    weights = weights / weights.sum()
    return str(rng.choice(names, p=weights))


def generate_load_scaling(seed, regime=None):
    rng = np.random.default_rng(int(seed))
    regime = regime or sample_load_regime(rng)
    spec = LOAD_SPECS[regime]
    base_net = pp.from_json_string(BASE_NET_JSON)
    load_table = _bus_load_table(base_net)
    base_bus_load = {int(idx): float(load_table.loc[idx, "p_load"]) for idx in load_table.index}
    load_buses = sorted(pd.unique(base_net.load["bus"].astype(int)))
    raw = _bounded_normal(rng, spec["mean"], spec["std"], spec["clip"][0], spec["clip"][1], len(load_buses))
    adjusted = _enforce_total_load_bounds(raw, load_buses, base_bus_load, spec["total_bounds"], spec["clip"])
    scaling = pd.Series(1.0, index=base_net.bus.index, dtype=float)
    for bus_idx, scale in zip(load_buses, adjusted):
        scaling.loc[int(bus_idx)] = float(scale)
    total_ratio = float(np.sum(adjusted * np.array([base_bus_load[int(bus)] for bus in load_buses])) / max(sum(base_bus_load.values()), 1e-9))
    return scaling.to_numpy(dtype=np.float32), {"regime": regime, "total_load_ratio": total_ratio, "scale_bounds": list(spec["clip"])}


def apply_load_scaling(net, scaling):
    scaling = np.asarray(scaling, dtype=float)
    for load_idx, row in net.load.iterrows():
        scale = float(scaling[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale
    return net


## 4. Power Flow, Constraints, Features, and Ranking Labels


In [ ]:
def _has_powerflow_initialization(net):
    return hasattr(net, "res_bus") and len(net.res_bus) == len(net.bus) and "vm_pu" in net.res_bus.columns


def run_powerflow(net):
    if not validate_configuration(net):
        return {"success": False, "reason": "invalid_topology"}
    init_mode = "results" if _has_powerflow_initialization(net) else "flat"
    try:
        pp.runpp(net, algorithm="nr", init=init_mode, calculate_voltage_angles=False, enforce_q_lims=False, max_iteration=50, tolerance_mva=1e-8, numba=False)
    except Exception as first_exc:
        try:
            pp.runpp(net, algorithm="nr", init="flat", calculate_voltage_angles=False, enforce_q_lims=False, max_iteration=50, tolerance_mva=1e-8, numba=False)
        except Exception as second_exc:
            return {"success": False, "reason": f"powerflow_failed: {type(second_exc).__name__}: {second_exc}"}
    if not bool(getattr(net, "converged", False)):
        return {"success": False, "reason": "powerflow_not_converged"}
    loss_kw = float(np.nan_to_num(net.res_line["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)
    vm = np.nan_to_num(net.res_bus["vm_pu"].to_numpy(dtype=float), nan=np.nan)
    loading = np.nan_to_num(net.res_line["loading_percent"].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    voltage_violations = int(((vm < VOLTAGE_NORMAL_MIN) | (vm > VOLTAGE_MAX)).sum())
    overload_count = int((loading > LOADING_HARD_MAX).sum())
    return {
        "success": True,
        "total_loss_kw": loss_kw,
        "min_voltage_pu": float(np.nanmin(vm)),
        "max_voltage_pu": float(np.nanmax(vm)),
        "max_line_loading_percent": float(np.nanmax(loading)),
        "voltage_violation_count": voltage_violations,
        "overload_count": overload_count,
        "constraint_violation_count": int(voltage_violations + overload_count),
    }


def candidate_is_valid(pf):
    if not pf.get("success", False):
        return False, pf.get("reason", "powerflow_failed")
    if float(pf["min_voltage_pu"]) < VOLTAGE_HARD_MIN:
        return False, "min_voltage_hard_reject"
    if float(pf["max_voltage_pu"]) > VOLTAGE_MAX:
        return False, "max_voltage_hard_reject"
    if float(pf["max_line_loading_percent"]) > LOADING_HARD_MAX:
        return False, "loading_hard_reject"
    if float(pf["total_loss_kw"]) > LOSS_HARD_MAX_KW:
        return False, "loss_hard_reject"
    if float(pf["total_loss_kw"]) < MIN_REALISTIC_LOSS_KW:
        return False, "loss_low_outlier"
    return True, "accepted"


def extract_graph_features(net, topology_record):
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    distance = compute_distance_to_substation(active_graph, root_bus)
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    node_features = []
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        node_features.append([
            float(load_table.loc[bus_idx, "p_load"]),
            float(load_table.loc[bus_idx, "q_load"]),
            float(depth.get(bus_idx_int, -1)),
            float(distance.get(bus_idx_int, np.inf)),
            1.0 if bus_idx_int == root_bus else 0.0,
            float(voltage_table.loc[bus_idx]),
        ])
    edge_features = []
    for line_idx, _ in net.line.iterrows():
        r, x = _line_impedance(net, line_idx)
        edge_features.append([float(r), float(x), 1.0 if line_is_closed(net, line_idx) else 0.0, float(np.sqrt(r * r + x * x))])
    total_load = float(load_table["p_load"].sum())
    global_features = np.array([total_load, float(len(net.bus)), float(len(net.line)), float(sum(1 for idx in net.line.index if line_is_closed(net, idx))), float(topology_record["topology_deviation"])], dtype=np.float32)
    return np.asarray(node_features, dtype=np.float32), np.asarray(edge_features, dtype=np.float32), global_features


def evaluate_candidate(load_scaling, topology_record):
    net = pp.from_json_string(BASE_NET_JSON)
    _apply_line_status(net, topology_record["line_status"])
    apply_load_scaling(net, load_scaling)
    pf = run_powerflow(net)
    valid, reason = candidate_is_valid(pf)
    if not valid:
        return None, reason
    node_features, edge_features, global_features = extract_graph_features(net, topology_record)
    labels = np.array([
        pf["total_loss_kw"],
        pf["min_voltage_pu"],
        pf["max_line_loading_percent"],
        pf["voltage_violation_count"],
        pf["overload_count"],
        pf["constraint_violation_count"],
    ], dtype=np.float32)
    return {
        "node_features": node_features,
        "edge_features": edge_features,
        "global_features": global_features,
        "labels": labels,
        "topology_id": topology_record["topology_id"],
        "switch_config_key": topology_record["switch_config_key"],
        "topology_deviation": int(topology_record["topology_deviation"]),
        "switch_operations": topology_record["switch_operations"],
    }, "accepted"


## 5. Scenario Generation and HDF5 Streaming


In [ ]:
def initialize_base_network():
    global BASE_NET_JSON, BASE_LINE_STATUS, BASE_TOTAL_LOAD_MW, BASE_POWERFLOW_RESULT, TOPOLOGY_POOL
    base_net = load_ieee33()
    BASE_LINE_STATUS = _line_status_vector(base_net)
    BASE_TOTAL_LOAD_MW = float(_bus_load_table(base_net)["p_load"].sum())
    BASE_POWERFLOW_RESULT = run_powerflow(base_net)
    if not BASE_POWERFLOW_RESULT.get("success", False):
        raise RuntimeError(f"Base power flow failed: {BASE_POWERFLOW_RESULT}")
    BASE_NET_JSON = pp.to_json(base_net)
    TOPOLOGY_POOL = precompute_topology_pool(base_net, TOPOLOGY_POOL_TARGET, GLOBAL_SEED)
    print(f"Base cached | load={BASE_TOTAL_LOAD_MW:.3f} MW | min_v={BASE_POWERFLOW_RESULT['min_voltage_pu']:.4f} | topology_pool={len(TOPOLOGY_POOL)}")


def generate_scenario(seed, candidates_per_scenario=CANDIDATES_PER_SCENARIO):
    rng = np.random.default_rng(int(seed))
    load_scaling, load_metadata = generate_load_scaling(int(seed) + 17)
    candidate_indices = [0]
    if len(TOPOLOGY_POOL) > 1:
        extra = rng.choice(np.arange(1, len(TOPOLOGY_POOL)), size=min(len(TOPOLOGY_POOL) - 1, candidates_per_scenario * 4), replace=False)
        candidate_indices.extend([int(x) for x in extra])
    candidate_records = []
    failure_reasons = {}
    for topology_index in candidate_indices:
        record, reason = evaluate_candidate(load_scaling, TOPOLOGY_POOL[int(topology_index)])
        if record is None:
            failure_reasons[reason] = int(failure_reasons.get(reason, 0)) + 1
            continue
        record["topology_index"] = int(topology_index)
        candidate_records.append(record)
        if len(candidate_records) >= candidates_per_scenario:
            break
    if len(candidate_records) < max(3, candidates_per_scenario // 2):
        return {"success": False, "seed": int(seed), "reason": "too_few_valid_candidates", "failure_reasons": failure_reasons}
    losses = np.array([float(record["labels"][0]) for record in candidate_records], dtype=float)
    order = np.argsort(losses)
    ranks = np.empty(len(losses), dtype=np.int32)
    ranks[order] = np.arange(1, len(losses) + 1, dtype=np.int32)
    is_best = np.zeros(len(losses), dtype=np.int8)
    is_best[int(order[0])] = 1
    base_loss = float(candidate_records[0]["labels"][0])
    best_loss = float(losses[int(order[0])])
    scenario_features = np.array([
        float(candidate_records[0]["global_features"][0]),
        float(load_metadata["total_load_ratio"]),
        float(REGIME_CODE[load_metadata["regime"]]),
    ], dtype=np.float32)
    return {
        "success": True,
        "seed": int(seed),
        "load_scaling_vector": load_scaling.astype(np.float32),
        "scenario_features": scenario_features,
        "candidate_node_features": np.stack([record["node_features"] for record in candidate_records]).astype(np.float32),
        "candidate_edge_features": np.stack([record["edge_features"] for record in candidate_records]).astype(np.float32),
        "candidate_global_features": np.stack([record["global_features"] for record in candidate_records]).astype(np.float32),
        "candidate_labels": np.stack([record["labels"] for record in candidate_records]).astype(np.float32),
        "candidate_ranks": ranks,
        "is_best_candidate": is_best,
        "topology_indices": np.array([record["topology_index"] for record in candidate_records], dtype=np.int32),
        "metadata": {
            "seed": int(seed),
            "load_metadata": load_metadata,
            "candidate_topology_ids": [record["topology_id"] for record in candidate_records],
            "candidate_switch_config_keys": [record["switch_config_key"] for record in candidate_records],
            "candidate_topology_deviation": [int(record["topology_deviation"]) for record in candidate_records],
            "candidate_switch_operations": [record["switch_operations"] for record in candidate_records],
            "num_candidates": int(len(candidate_records)),
            "best_candidate_position": int(order[0]),
            "best_topology_index": int(candidate_records[int(order[0])]["topology_index"]),
            "base_loss_kw": base_loss,
            "best_loss_kw": best_loss,
            "loss_improvement_kw": float(base_loss - best_loss),
            "failure_reasons": failure_reasons,
        },
    }


def initialize_hdf5(path, overwrite=True):
    mode = "w" if overwrite else "a"
    with h5py.File(path, mode, track_order=True) as h5:
        h5.attrs["schema_version"] = "dnr_candidate_ranking_v1"
        h5.attrs["dataset_name"] = "ieee33_dnr_candidate_ranking"
        h5.attrs["global_seed"] = int(GLOBAL_SEED)
        h5.attrs["n_scenarios_target"] = int(N_SCENARIOS)
        h5.attrs["candidates_per_scenario"] = int(CANDIDATES_PER_SCENARIO)
        h5.attrs["node_feature_names"] = json.dumps(NODE_FEATURE_NAMES)
        h5.attrs["edge_feature_names"] = json.dumps(EDGE_FEATURE_NAMES)
        h5.attrs["global_feature_names"] = json.dumps(GLOBAL_FEATURE_NAMES)
        h5.attrs["label_names"] = json.dumps(LABEL_NAMES)
        h5.attrs["scenario_feature_names"] = json.dumps(SCENARIO_FEATURE_NAMES)
        h5.attrs["load_regime_policy"] = json.dumps(LOAD_REGIME_POLICY)
        h5.attrs["topology_pool_size"] = int(len(TOPOLOGY_POOL) if TOPOLOGY_POOL is not None else 0)
        h5.attrs["attempted_scenarios"] = 0
        h5.attrs["valid_scenarios"] = 0
        h5.attrs["rejected_scenarios"] = 0
        h5.attrs["rejection_reasons"] = json.dumps({})
    return path


def _write_array(group, name, value):
    group.create_dataset(name, data=np.asarray(value), compression=COMPRESSION, compression_opts=COMPRESSION_OPTS, shuffle=True)


def write_scenarios_to_hdf5(path, records, start_id):
    string_dtype = h5py.string_dtype(encoding="utf-8")
    next_id = int(start_id)
    with h5py.File(path, "a", track_order=True) as h5:
        for record in records:
            group = h5.create_group(f"scenario_{next_id:06d}")
            _write_array(group, "load_scaling_vector", record["load_scaling_vector"])
            _write_array(group, "scenario_features", record["scenario_features"])
            _write_array(group, "candidate_node_features", record["candidate_node_features"])
            _write_array(group, "candidate_edge_features", record["candidate_edge_features"])
            _write_array(group, "candidate_global_features", record["candidate_global_features"])
            _write_array(group, "candidate_labels", record["candidate_labels"])
            _write_array(group, "candidate_ranks", record["candidate_ranks"])
            _write_array(group, "is_best_candidate", record["is_best_candidate"])
            _write_array(group, "topology_indices", record["topology_indices"])
            group.create_dataset("metadata", data=json.dumps(record["metadata"]), dtype=string_dtype)
            group.attrs["seed"] = int(record["seed"])
            group.attrs["num_candidates"] = int(record["metadata"]["num_candidates"])
            group.attrs["best_topology_index"] = int(record["metadata"]["best_topology_index"])
            group.attrs["base_loss_kw"] = float(record["metadata"]["base_loss_kw"])
            group.attrs["best_loss_kw"] = float(record["metadata"]["best_loss_kw"])
            group.attrs["loss_improvement_kw"] = float(record["metadata"]["loss_improvement_kw"])
            next_id += 1
    return next_id


def update_hdf5_report(path, state):
    with h5py.File(path, "a") as h5:
        h5.attrs["attempted_scenarios"] = int(state["attempted"])
        h5.attrs["valid_scenarios"] = int(state["valid"])
        h5.attrs["rejected_scenarios"] = int(state["rejected"])
        h5.attrs["rejection_reasons"] = json.dumps(state["rejection_reasons"])


def generate_dataset(n_scenarios=N_SCENARIOS, h5_path=H5_PATH, batch_size=BATCH_SIZE, n_jobs=N_JOBS, overwrite=True):
    initialize_base_network()
    h5_path = initialize_hdf5(h5_path, overwrite=overwrite)
    next_seed = int(GLOBAL_SEED)
    next_id = 1
    state = {"attempted": 0, "valid": 0, "rejected": 0, "rejection_reasons": {}}
    progress = tqdm(total=n_scenarios, desc="Accepted ranking scenarios", unit="scenario")
    max_attempts = int(n_scenarios * 4)
    while state["valid"] < n_scenarios and state["attempted"] < max_attempts:
        remaining = n_scenarios - state["valid"]
        current_batch = min(batch_size, max(remaining * 2, 1))
        seeds = [next_seed + i for i in range(current_batch)]
        next_seed += current_batch
        results = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(generate_scenario)(seed, CANDIDATES_PER_SCENARIO) for seed in seeds
        )
        accepted = []
        for result in results:
            if state["valid"] >= n_scenarios:
                break
            state["attempted"] += 1
            if result.get("success", False):
                accepted.append(result)
                state["valid"] += 1
            else:
                state["rejected"] += 1
                reason = result.get("reason", "unknown")
                state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1
        if accepted:
            next_id = write_scenarios_to_hdf5(h5_path, accepted, next_id)
            progress.update(len(accepted))
        update_hdf5_report(h5_path, state)
        print(f"batch status | attempted={state['attempted']} valid={state['valid']} rejected={state['rejected']} reasons={state['rejection_reasons']}")
    progress.close()
    if state["valid"] < n_scenarios:
        raise RuntimeError(f"Only generated {state['valid']} of {n_scenarios}. Rejections={state['rejection_reasons']}")
    report = {
        "h5_path": h5_path,
        "attempted_scenarios": int(state["attempted"]),
        "valid_scenarios": int(state["valid"]),
        "rejected_scenarios": int(state["rejected"]),
        "acceptance_rate_percent": float(100.0 * state["valid"] / max(state["attempted"], 1)),
        "topology_pool_size": int(len(TOPOLOGY_POOL)),
        "candidates_per_scenario": int(CANDIDATES_PER_SCENARIO),
        "rejection_reasons": state["rejection_reasons"],
    }
    print(json.dumps(report, indent=2))
    return report


ranking_generation_report = generate_dataset(N_SCENARIOS, H5_PATH, BATCH_SIZE, N_JOBS, overwrite=True)


## 6. Validation and Analysis


In [ ]:
def _read_metadata_dataset(dataset):
    raw = dataset[()]
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def collect_analysis(h5_path=H5_PATH):
    rows = []
    with h5py.File(h5_path, "r") as h5:
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        names = sorted(name for name in h5.keys() if name.startswith("scenario_"))
        for name in tqdm(names, desc="Reading scenarios", unit="scenario"):
            group = h5[name]
            labels = group["candidate_labels"][:]
            ranks = group["candidate_ranks"][:]
            topology_indices = group["topology_indices"][:]
            metadata = _read_metadata_dataset(group["metadata"])
            best_pos = int(np.argmin(ranks))
            for i in range(labels.shape[0]):
                rows.append({
                    "scenario": name,
                    "candidate": int(i),
                    "topology_index": int(topology_indices[i]),
                    "rank": int(ranks[i]),
                    "is_best": int(i == best_pos),
                    "loss_kw": float(labels[i, 0]),
                    "min_voltage_pu": float(labels[i, 1]),
                    "max_loading_percent": float(labels[i, 2]),
                    "violations": int(labels[i, 5]),
                    "load_regime": metadata["load_metadata"]["regime"],
                    "loss_improvement_kw": float(metadata["loss_improvement_kw"]),
                })
    return pd.DataFrame(rows), attrs


def print_quality_report(df, attrs):
    scenario_count = df["scenario"].nunique()
    print("Ranking Dataset Quality Report")
    print("-" * 34)
    print(f"Scenarios              : {scenario_count:,}")
    print(f"Candidate rows         : {len(df):,}")
    print(f"Topology pool size     : {int(attrs.get('topology_pool_size', 0))}")
    print(f"Used topology count    : {df['topology_index'].nunique()}")
    print(f"Acceptance rate        : {100.0 * int(attrs.get('valid_scenarios', 0)) / max(int(attrs.get('attempted_scenarios', 1)), 1):.2f}%")
    print(f"Mean best improvement  : {df[df['is_best'] == 1]['loss_improvement_kw'].mean():.3f} kW")
    print(f"Max candidate loss     : {df['loss_kw'].max():.3f} kW")
    print(f"Min candidate voltage  : {df['min_voltage_pu'].min():.4f} pu")
    print(f"Max candidate loading  : {df['max_loading_percent'].max():.2f}%")
    print(f"Load regimes           : {df.drop_duplicates('scenario')['load_regime'].value_counts().to_dict()}")
    print(f"Rejection reasons      : {json.loads(attrs.get('rejection_reasons', '{}'))}")


def plot_loss_distribution(df):
    fig = px.histogram(df, x="loss_kw", color="rank", nbins=60, title="Candidate Loss Distribution by Rank")
    fig.show()


def plot_best_improvement(df):
    best = df[df["is_best"] == 1]
    fig = px.histogram(best, x="loss_improvement_kw", nbins=50, title="Loss Improvement of Best Candidate vs Base Candidate")
    fig.show()


def plot_topology_reuse(df):
    counts = df["topology_index"].value_counts().reset_index()
    counts.columns = ["topology_index", "count"]
    fig = px.bar(counts.head(80), x="topology_index", y="count", title="Candidate Topology Reuse")
    fig.show()


def plot_voltage_loading(df):
    fig = px.scatter(df, x="min_voltage_pu", y="max_loading_percent", color="rank", title="Voltage vs Loading by Candidate Rank")
    fig.show()


analysis_df, ranking_attrs = collect_analysis(H5_PATH)
print_quality_report(analysis_df, ranking_attrs)
plot_loss_distribution(analysis_df)
plot_best_improvement(analysis_df)
plot_topology_reuse(analysis_df)
plot_voltage_loading(analysis_df)
analysis_df.describe(include="all")


## 7. Final Dataset Summary


In [ ]:
def print_dataset_summary(h5_path=H5_PATH):
    with h5py.File(h5_path, "r") as h5:
        names = sorted(name for name in h5.keys() if name.startswith("scenario_"))
        if len(names) == 0:
            raise RuntimeError("No scenarios found in HDF5 file.")
        first = h5[names[0]]
        print("DNR Candidate-Ranking Dataset Ready")
        print("=" * 42)
        print(f"HDF5 path                 : {h5_path}")
        print(f"Scenarios                 : {len(names):,}")
        print(f"Candidates per scenario    : {first['candidate_labels'].shape[0]}")
        print(f"Topology pool size         : {int(h5.attrs.get('topology_pool_size', 0))}")
        print(f"candidate_node_features    : {first['candidate_node_features'].shape}")
        print(f"candidate_edge_features    : {first['candidate_edge_features'].shape}")
        print(f"candidate_global_features  : {first['candidate_global_features'].shape}")
        print(f"candidate_labels           : {first['candidate_labels'].shape}")
        print(f"Labels                     : {json.loads(h5.attrs['label_names'])}")
        print("")
        print("Training objective")
        print("- Train a model to score each candidate topology for a fixed load scenario.")
        print("- Select the candidate with minimum predicted loss subject to voltage/loading constraints.")
        print("- Avoid direct imbalanced topology-class classification.")


print_dataset_summary(H5_PATH)
